- https://yingru.notion.site/When-Speed-Kills-Stability-271211a558b7808d8b12d403fd15edda

### $\pi_\theta$, $\pi_{old}$, $\pi_{ref}$

- `ActorRolloutRefWorker`
    - This worker can be instantiated as a standalone actor or a standalone rollout or a standalone reference policy or a hybrid engine based on the config.rollout

- 逻辑上
    - Policy 是 Role.ActorRollout 同一个 worker 里既负责 actor update，也负责 rollout
    - Ref 是 Role.RefPolicy （仅在开 KL reward / KL loss 时才创建）
- 物理部署上
    - ActorRollout 和 RefPolicy 都映射到 global_pool （Colocation (共生/同驻)）
        - `self.mapping[Role.ActorRollout] = global_pool_id`
        - `self.mapping[Role.Critic] = global_pool_id`
        - `self.mapping[Role.RefPolicy] = "global_pool"`

- 计算
    - Actor 与 Rollout 的切换 (Hybrid Engine)：`fsdp_workers.py`
        - rollout_mode: 切换到生成模式
        - trainer_mode(): 切换到训练模式
    - RefPolicy 通常是一个冻结的模型，不需要反向传播。为了节省显存给 Actor 训练，RefPolicy 采用了 FSDP CPU Offload 策略。
        - 当需要计算 Ref Log Prob 时，FSDP 机制会自动将需要的参数分片（Shard）流式传输（Stream）到 GPU 进行前向计算，计算完后立即释放 GPU 显存。
- 计算时机
    - 同一 batch 多轮 PPO 更新导致“批内陈旧”：π_old 在 batch 开头算一次，但 π_θ 会随 ppo_epochs/mini-batch 迭代偏移，后续更新天然更 off-policy。

### actor 与 ref

- Actor (训练主角)：常驻 GPU。
    - 因为它需要频繁进行 Forward（计算 LogProb）、Backward（计算梯度）和 Optimizer Step（更新参数）。
    - 如果频繁在 CPU/GPU 间搬运它，训练速度会慢得无法接受。
- Ref Model (训练配角)：常驻 CPU。
    - 在 verl/workers/fsdp_workers.py 初始化时，Ref 被强制开启了 cpu_offload=True。
        - `ref.fsdp_config.param_offload=True`，而且 ref 是 forward_only=True
    - 这意味着它的本体一直都在 CPU 内存里。
- 调用 compute_ref_log_prob 时 （`FSDP(cpu_offload=cpu_offload)`）
    - 流式加载 (Streaming)：FSDP 框架接管控制权。当计算进行到 Ref 模型的第 i 层时，FSDP 会自动把第 i 层的参数从 CPU 拷贝到 GPU。
    - 即用即弃：第 i 层计算完毕后，FSDP 会立即释放这一层在 GPU 上的显存（参数被销毁，只保留输出的 Activation）。
    - 无缝衔接：整个计算过程中，GPU 上同一时刻只有 Actor 的全部参数 + Ref 模型的一层参数。

```mermaid
sequenceDiagram
      autonumber
      participant Driver as Trainer Loop
      participant Policy as π_theta / π_old
      participant Ref as π_ref
      participant Rollout as π_rollout
      participant GPU as One Physical GPU

      Note over GPU: 显存 = 常驻底座 + 当前相位峰值

      rect rgb(245,245,245)
      Note over Policy,GPU: 1. Actor steady base
      Note over Policy,GPU: optimizer states + grad buffers + CUDA/NCCL/runtime
      end

      Driver->>Rollout: generate_sequences()
      activate Rollout
      Note over Rollout,GPU: 2. π_rollout steady peak
      Note over Rollout,GPU: vLLM weights + KV cache + decode/prefill buffers + MM encoder activations
      Note over GPU: 此时总显存 ≈ Actor steady base + π_rollout steady peak
      Rollout-->>Driver: sampled responses
      deactivate Rollout

      Driver->>Rollout: sleep_replicas()
      activate Rollout
      Note over Rollout,GPU: vLLM sleep，释放 weights + KV cache 大头
      Rollout-->>Driver: rollout memory reduced
      deactivate Rollout

      Driver->>Policy: compute_old_log_prob()
      activate Policy
      Note over Policy,GPU: actor model shard 上卡，做 old_log_prob forward
      Note over GPU: 此时总显存 ≈ Actor steady base + actor forward peak
      Policy-->>Driver: old_log_probs
      deactivate Policy

      Driver->>Ref: compute_ref_log_prob()
      activate Ref
      Note over Ref,GPU: 3. π_ref forward peak
      Note over Ref,GPU: ref model shard 上卡，仅 forward，无 optimizer/grad 大底座
      Note over Ref,GPU: 主要是 ref shard + MM encoder activations + FSDP comm/reshard buffers
      Note over GPU: 此时总显存 ≈ Actor steady base + π_ref forward peak
      Ref-->>Driver: ref_log_probs
      deactivate Ref

      Driver->>Policy: update_actor()
      activate Policy
      Note over Policy,GPU: actor training peak
      Note over Policy,GPU: model shard + optimizer + grad + backward activations
      Note over GPU: 这是训练相位的最高峰之一
      Policy-->>Driver: updated weights
      deactivate Policy

      Driver->>Policy: update_weights()
      activate Policy
      Driver->>Rollout: resume(weights)
      activate Rollout
      Note over GPU: 切换相位：短时可能出现 trainer + rollout 同时偏高
      Policy-->>Rollout: stream latest weights
      Policy->>Policy: offload model only
      Rollout->>Rollout: resume(kv_cache)
      deactivate Rollout
      deactivate Policy

      Driver->>Rollout: next generate_sequences()
      activate Rollout
      Note over GPU: 回到 rollout 稳态<br/>Actor steady base + π_rollout steady peak
      Rollout-->>Driver: next batch
      deactivate Rollout
```